# scRNA-seq + TCR – 05 CD62L/KLF2 populations of naive CD4 T cells (mouse)

10x Genomics GEX + V(D)J (TCR) + hashtag (HTO) data set `SM_EBA_multi` (Cell Ranger multi). Four sorted
populations (CD62L⁻KLF2⁻ "Tfh-like", CD62L⁻KLF2⁺ "Th1-like", CD62L⁺KLF2⁻ "Undiff 2", CD62L⁺KLF2⁺ "Undiff 1")
were hashtagged and pooled in one library.

**Input**
* Cell Ranger multi output: `sample_filtered_feature_bc_matrix/` (GEX + antibody capture) and `vdj_t/filtered_contig_annotations.csv`
* Reference for label transfer: annotated T cells of the adoptive-transfer data set (output of `04_Tcell_annotation_analysis.Rmd`, Unknowns removed)

**Output:** `SM_EBA_hashtag_assignment_per_cell.csv`, figures in `fig_dir`

Steps
1. GEX QC (300 < genes < 7,500, < 10 % mitochondrial), HTO demultiplexing (`HTODemux`, singlets only)
2. TCR clonotypes (scRepertoire `combineTCR`, multi-chain cells kept); dual TCR = > 1 TRA **or** > 1 TRB chain
3. Dual vs single TCR per population, genes differing between dual- and single-TCR cells, clone sizes
4. Projection onto the reference UMAP and label transfer of `Celltype_2` (`FindTransferAnchors` / `MapQuery`)

**Figure panels produced here** (panel as file prefix)

| panel | content | output |
|---|---|---|
| Fig. 5d | reference UMAP coloured by transferred `Celltype_2` | `Fig5d_refUMAP_predicted_celltypes.pdf` |
| Fig. 5e | reference UMAP coloured by CD62L/KLF2 population | `Fig5e_refUMAP_CD62L_KLF2_populations.pdf` |
| Fig. 5f | proportion of the four populations per predicted cell type | `Fig5f_celltypes_by_population_proportion.pdf` |
| Fig. 6a | fraction of dual-TCR cells per population | `Fig6a_dualTCR_fraction_per_population.pdf` |
| Fig. 6b | clone sizes per population (largest clone removed) | `Fig6b_clone_sizes_per_population.pdf` |
| Fig. 6c | dual-TCR cells per predicted cell type and population | `Fig6c_dualTCR_celltypes_by_population_counts.pdf` |
| Fig. 6d (left) | marker dot plot, CD62L⁻ populations, single vs dual TCR | `Fig6d_left_DotPlot_CD62Lneg_single_vs_dualTCR.pdf` |
| Suppl. Fig. 5a | top-10 genes of dual- vs single-TCR cells per population | `SuppFig5a_DotPlot_dualTCR_DEGs_per_population.pdf` |

Fig. 6d (right, day-6 Tfh vs Th1) comes from `04_Tcell_annotation_analysis.Rmd`.

## 1. Setup

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(patchwork)
library(scRepertoire)
library(stringr)
library(DT)
library(tidyr)
library(tidyverse)

In [ ]:
# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
multi_dir <- "../data/SM_EBA_multi/outs/per_sample_outs/SM_EBA_multi/"   # Cell Ranger multi output
filtered_path <- paste0(multi_dir, "count/sample_filtered_feature_bc_matrix/")
vdj_path      <- paste0(multi_dir, "vdj_t/filtered_contig_annotations.csv")

# Reference: annotated T cells (output of 04_Tcell_annotation_analysis.Rmd)
ref_rds <- "../RData/RDS_Tcells_annotated.rds"

fig_dir   <- "../results/figures/"
table_dir <- "../results/tables/"
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)
dir.create(table_dir, showWarnings = FALSE, recursive = TRUE)

In [ ]:
cols_samples <- c(
  "CD62L-KLF2-" = "#006d2c",  # green  ("Tfh-like")
  "CD62L-KLF2+" = "#fe9929",  # orange ("Th1-like")
  "CD62L+KLF2-" = "#f03b20",  # red    ("Undiff 2")
  "CD62L+KLF2+" = "#253494"   # blue   ("Undiff 1")
)

condition_labels <- c("CD62L-KLF2-", "CD62L-KLF2+", "CD62L+KLF2-", "CD62L+KLF2+")

Colours of the reference annotation (as in 04).

In [ ]:
celltype_colors_2 <- c(
  "Th0 I" = "#a1d99b",               # green
  "Th0 II" = "#74c476",
  "Th0 III" = "#41ab5d",
  "Th0 IV" = "#238b45",
  "Th0 V" = "#006d2c",
  "Th0 VI" = "#00441b",

  "Tfh" = "#08519c",                   # blue
  "Pre-Tfh I" = "#3182bd",
  "Pre-Tfh II" = "#6baed6",
  "Pre-Tfh proliferating" = "#bdd7e7", #new!

  "Th1" = "#a50f15",
  "Pre-Th1 I" = "#de2d26",             # orange !!!# red
  "Pre-Th1 II" = "#fb6a4a",
  "Pre-Th1 proliferating" = "#fcae91",

  "Unknown 1" = "#bdbdbd",             # gray
  "Unknown 2" = "#969696",
  "Unknown 3" = "#636363"
)

celltype_colors_3 <- c(
  "Th0 I" = "#a1d99b",               # green
  "Th0 II" = "#74c476",
  "Th0 III" = "#41ab5d",
  "Th0 IV" = "#238b45",
  "Th0 V" = "#006d2c",
  "Th0 VI" = "#00441b",

  "Tfh" = "#08519c",                   # blue
  "Pre-Tfh I" = "#3182bd",
  "Pre-Tfh II" = "#6baed6",
  "Pre-Tfh proliferating" = "#bdd7e7", #new!

  "Th1" = "#a50f15",
  "Pre-Th1 I" = "#de2d26",             # orange !!!# red
  "Pre-Th1 II" = "#fb6a4a",
  "Pre-Th1 proliferating" = "#fcae91"
)

## 2. Gene expression: read-in and QC

In [ ]:
# Read matrix (returns a list: GEX + HTO)
mat <- Read10X(filtered_path)

# Extract layers
gex <- mat$`Gene Expression`
hto <- mat$`Antibody Capture`

# Create object
seu <- CreateSeuratObject(counts = gex, assay = "RNA", min.cells = 3, min.features = 200)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 3.5)

# RNA QC metrics
seu[["percent.mt"]] <- PercentageFeatureSet(seu, pattern = "^mt-")
# Inspect distributions
VlnPlot(seu, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), pt.size = 0)

In [ ]:
options(repr.plot.width = 3, repr.plot.height = 4)

VlnPlot(seu, features = c("nFeature_RNA"), pt.size = 0) + geom_hline(yintercept = 300, linetype = "dashed", color = "red")
VlnPlot(seu, features = c( "percent.mt"), pt.size = 0) + geom_hline(yintercept = 10, linetype = "dashed", color = "red")

In [ ]:
# GEX cell filter
gex_cells <- colnames(seu)[
  seu$nFeature_RNA > 300 &
  seu$nFeature_RNA < 7500 &
  seu$percent.mt < 10
]

## 3. Hashtags: demultiplexing of the four sorted populations

Cells passing the GEX filter and with ≥ 1 HTO count; CLR normalisation and `HTODemux`.
`Condition` = hashtag name (`hash.ID`) converted to `CD62L±KLF2±`.

In [ ]:
# Barcodes of GEX and HTO already match - subset to be safe
hto <- hto[, colnames(seu), drop = FALSE]

# Add HTO assay
seu[["HTO"]] <- CreateAssayObject(counts = hto)

# HTO-positive cells
hto_cells <- colnames(seu)[Matrix::colSums(seu[["HTO"]]@counts) > 0]

# Intersection of RNA- and HTO-filtered cells
keep_cells <- intersect(gex_cells, hto_cells)

length(keep_cells)

# Subset object
seu <- subset(seu, cells = keep_cells)

seu

In [ ]:
# Normalise HTO and demultiplex
seu <- NormalizeData(seu, assay = "HTO", normalization.method = "CLR")
seu <- HTODemux(seu, assay = "HTO")

# Inspect HTO assignments
table(seu$HTO_classification.global)

In [ ]:
seu@meta.data$Condition <- seu@meta.data$hash.ID
seu@meta.data$Condition <- gsub("pos", "+", seu@meta.data$Condition)
seu@meta.data$Condition <- gsub("-", "", seu@meta.data$Condition)
seu@meta.data$Condition <- gsub("neg", "-", seu@meta.data$Condition)
head(seu@meta.data)
table(seu@meta.data$Condition)

In [ ]:
assign <- data.frame(
  barcode        = colnames(seu),
  hash_ID        = seu$hash.ID,                   # hashtag / population (or Doublet / Negative)
  classification = seu$HTO_classification.global, # Singlet / Doublet / Negative
  HTO_maxID      = seu$HTO_maxID,
  HTO_margin     = seu$HTO_margin
)
write.csv(assign, paste0(table_dir, "SM_EBA_hashtag_assignment_per_cell.csv"), row.names = FALSE)

## 4. TCR clonotypes

In [ ]:
contigs <- read.csv(vdj_path)
head(contigs)
dim(contigs)

In [ ]:
# one library -> list with one element
contig_list <- list(sample1 = contigs)

# clonotypes per cell; multi-chain cells are kept (removeMulti = FALSE)
combined <- combineTCR(contig_list, samples = "sample1", ID = "sample1",removeNA = TRUE,
                           removeMulti = FALSE)

tcr <- combined[[1]]

In [ ]:
combined[[1]]$barcode <- gsub(".*_", "", combined[[1]]$barcode)

In [ ]:
cols_to_add <- c(
    "sample", "ID",
    "TCR1", "cdr3_aa1", "cdr3_nt1",
    "TCR2", "cdr3_aa2", "cdr3_nt2",
    "CTgene", "CTnt", "CTaa", "CTstrict",
    "cellType"
)
tcr$barcode <- gsub(".*_", "", tcr$barcode)
rownames(tcr) <- tcr$barcode
seu <- AddMetaData(seu, tcr[, cols_to_add])

In [ ]:
tcr_meta <- seu@meta.data %>%
  select(Condition = HTO_classification,
         CTgene, CTstrict, TCR1, TCR2, cdr3_aa1, cdr3_aa2)

tcr_class <- tcr_meta %>%
  mutate(
    # 1) dual when multiple chains present
    dual_TCR = str_detect(CTgene, ";"),

    # 2) productive clonotype: CTgene exists AND CTstrict exists
    productive = !is.na(CTgene) & !is.na(CTstrict),

    # 3) single productive TCR (exactly one TRA + one TRB)
    single_TCR = productive & !dual_TCR,

    # 4) nonproductive = everything else
    nonproductive = !productive
  )

# Summarise per condition
tcr_summary <- tcr_class %>%
  group_by(Condition) %>%
  summarise(
    single_TCR = sum(single_TCR, na.rm = TRUE),
    dual_TCR = sum(dual_TCR, na.rm = TRUE),
    nonproductive = sum(nonproductive, na.rm = TRUE),
    total_cells = n()
  )

tcr_summary

## 5. Singlets and UMAP

In [ ]:
# ---------------------------------------------------------
# Keep only singlets and discard doublets/negatives
# ---------------------------------------------------------
seu_singlets <- subset(seu, subset = HTO_classification.global == "Singlet")
table(seu_singlets$Condition)

In [ ]:
# ---------------------------------------------------------
# Normalise RNA + UMAP
# ---------------------------------------------------------
seu_singlets <- NormalizeData(seu_singlets)
seu_singlets <- FindVariableFeatures(seu_singlets)
seu_singlets <- ScaleData(seu_singlets)
seu_singlets <- RunPCA(seu_singlets)
seu_singlets <- RunUMAP(seu_singlets, dims = 1:20)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4.5)
# ---------------------------------------------------------
# UMAP by population and by scRepertoire cell type
# ---------------------------------------------------------

# UMAP by biological condition (CD62L/KLF2 states)
DimPlot(seu_singlets, group.by = "Condition", label = FALSE, repel = TRUE, cols = cols_samples) + ggtitle("Barcodes")

DimPlot(seu_singlets, group.by = "cellType", label = FALSE)

## 6. Dual TCRs

Clonotypes with more than one chain are separated by `;` in `CTgene`. The most frequent clonotype
`TRAV14D-3-DV8.TRAJ15.TRAC_TRBV13-1.NA.TRBJ2-5.TRBC2` is the transgenic TCR.
Dual TCR = more than one TRA **or** more than one TRB chain.

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 5.5)

ct_counts <- as.data.frame(table(seu_singlets$CTgene)) %>%
  arrange(desc(Freq)) %>%
  rename(CTgene = Var1, count = Freq)

head(ct_counts)

N = 25   # top N clonotypes

ct_counts_top <- ct_counts %>%
  slice_max(count, n = N) %>%
  mutate(CTgene = fct_reorder(CTgene, count))

ggplot(ct_counts_top, aes(x = CTgene, y = count)) +
  geom_col(fill="steelblue", color="black") +
  labs(title = paste("Top", N, "\n most frequent \nclonotypes"),
       x="",
       y="Cell count") +
  theme_classic(base_size = 14) +
  coord_flip()

In [ ]:
ct <- seu_singlets$CTgene %>% replace_na("")
# split CTgene on both "_" (TRA_TRB) and ";" (additional chains)
parts <- str_split(ct, "[_;]")

seu_singlets$TRA_count <- sapply(parts, function(x) sum(str_detect(x, "^TRAV")))
seu_singlets$TRB_count <- sapply(parts, function(x) sum(str_detect(x, "^TRBV")))

# dual TCR = additional alpha OR additional beta chain
seu_singlets$dual_TCR <- seu_singlets$TRA_count > 1 | seu_singlets$TRB_count > 1

table(seu_singlets$dual_TCR)

In [ ]:
prop.table(table(seu_singlets$Condition, seu_singlets$dual_TCR),1)

**Figure 6a** – fraction of dual-TCR cells per population.

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 3)
df <- seu_singlets@meta.data %>%
  group_by(Condition, dual_TCR) %>%
  summarise(n = n(), .groups = "drop") %>%
  group_by(Condition) %>%
  mutate(freq = n / sum(n))

# stacked barplot (dual vs single)
p_fig6a <- ggplot(df, aes(x = Condition, y = freq, fill = dual_TCR)) +
  geom_col(width = 0.75, color="black") +
  scale_fill_manual(values = c("FALSE"="grey70","TRUE"="firebrick"),
                    labels = c("single TCR", "dual TCR"),
                    name = "TCR per cell") +
  labs(x = "", y = "Fraction of cells") +
  theme_classic(base_size = 14) +
  coord_flip()

In [ ]:
p_fig6a
ggsave(plot = p_fig6a, filename = paste0(fig_dir, "Fig6a_dualTCR_fraction_per_population.pdf"), width = 6, height = 3)

Number of clonotypes per population.

In [ ]:
clones_per_pop <- seu_singlets@meta.data %>%
  group_by(Condition) %>%
  summarise(
    unique_clones = n_distinct(CTgene),
    total_cells   = n(),
    diversity_ratio = unique_clones / total_cells
  )

clones_per_pop

In [ ]:
length(unique(seu_singlets@meta.data$CTgene))

## 7. Genes differing between dual- and single-TCR cells

Top-10 up-regulated genes (adj. P < 0.05) of dual-TCR vs other cells and vice versa, shown per population
→ **Supplementary Figure 5a**.

In [ ]:
seu_singlets@meta.data$dual_TCR_group <- ifelse(
  seu_singlets@meta.data$dual_TCR == TRUE,
  "dual TCR",
  "non-dual"
)

seu_singlets@meta.data$dual_TCR_group <- factor(
  seu_singlets@meta.data$dual_TCR_group,
  levels = c( "dual TCR", "non-dual")
)

table(seu_singlets@meta.data$dual_TCR_group)

Idents(seu_singlets) <- "dual_TCR_group"
dual_tcr_degs <- FindAllMarkers(
  object = seu_singlets,
  assay = "RNA",
  only.pos = TRUE,
  test.use = "wilcox",
  min.pct = 0.10,
  logfc.threshold = 0.25
)

top10_dual_tcr <- dual_tcr_degs %>%
  filter(p_val_adj < 0.05) %>%
  group_by(cluster) %>%
  slice_max(
    order_by = avg_log2FC,
    n = 10,
    with_ties = FALSE
  ) %>%
  ungroup()

top10_dual_tcr

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 3.5)
p1 <- DotPlot(
  seu_singlets,
  features = top10_dual_tcr$gene,
  group.by = "Condition"
) +
  RotatedAxis() +
  ggtitle("DEGs per HTO Condition")+

  # Color scale: low=blue, mid=white, high=red
  scale_colour_gradient2(
    low = "blue",
    mid = "white",
    high = "red",
    midpoint = 0
  ) +

  scale_size_continuous(range = c(0.5, 6)) +

  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 12),
    axis.text.y = element_text(size = 12),
    plot.title = element_text(size = 16)
  )

p1

ggsave(plot = p1, filename = paste0(fig_dir, "SuppFig5a_DotPlot_dualTCR_DEGs_per_population.pdf"),
       width =8, height = 3.5)

**Figure 6d (left)** – marker panel of Fig. 2b in the CD62L⁻ populations, split by single / dual TCR.

In [ ]:
marker_genes <- c("Mki67", "Il2ra", "Id2", "S100a4", "Tbx21", "Klf2", "Il21",
                  "Bcl6", "Icos", "Pdcd1", "Cxcr5", "Selplg", "S1pr1", "Ccr7")

cd62l_neg <- c("CD62L-KLF2+", "CD62L-KLF2-")

# subset: CD62L- populations only
seu_cd62l <- subset(
  seu_singlets,
  cells = rownames(seu_singlets@meta.data)[seu_singlets$Condition %in% cd62l_neg]
)

# group = population x TCR
seu_cd62l$TCR <- ifelse(seu_cd62l$dual_TCR, "dual TCR", "single TCR")
seu_cd62l$Cond_TCR <- factor(
  paste(seu_cd62l$Condition, seu_cd62l$TCR, sep = " | "),
  levels = c("CD62L-KLF2+ | single TCR", "CD62L-KLF2+ | dual TCR",
             "CD62L-KLF2- | single TCR", "CD62L-KLF2- | dual TCR")
)
table(seu_cd62l$Cond_TCR)   # cell numbers

marker_genes <- marker_genes[marker_genes %in% rownames(seu_cd62l)]

options(repr.plot.width = 7, repr.plot.height = 4.5)
p_cd62l <- DotPlot(
  seu_cd62l,
  features = rev(marker_genes),   # rev + coord_flip -> Mki67 on top (as in Fig. 2b)
  group.by = "Cond_TCR"
) +
  coord_flip() +
  RotatedAxis() +
  ggtitle("CD62L- cells: single vs dual TCR") +
  scale_colour_gradient2(low = "blue", mid = "white", high = "red", midpoint = 0) +
  scale_size_continuous(range = c(0.5, 6)) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 12),
    axis.text.y = element_text(size = 12, face = "italic"),
    plot.title  = element_text(size = 14)
  ) +
  xlab(NULL) + ylab(NULL)

p_cd62l

ggsave(plot = p_cd62l, filename = paste0(fig_dir, "Fig6d_left_DotPlot_CD62Lneg_single_vs_dualTCR.pdf"),
       width = 7, height = 4.5)

## 8. Clone sizes per population → Figure 6b

Clones with ≥ 3 cells in at least one population; the largest clone (transgenic TCR) is removed.
Point size = clone size capped at 50, colour = actual clone size, dual-TCR clones first.

In [ ]:
options(repr.plot.width = 9.5, repr.plot.height = 7.5)

# 1) Clone size summary
clone_summary <- seu_singlets@meta.data %>%
  filter(!is.na(CTgene)) %>%
  group_by(CTgene) %>%
  summarise(
    clone_size = n(),
    dual_TCR   = first(dual_TCR),
    .groups    = "drop"
  ) %>%
  arrange(desc(clone_size))

# 2) Identify largest clone to remove
largest_clone <- clone_summary %>%
  slice_max(clone_size, n = 1, with_ties = FALSE) %>%
  pull(CTgene)

largest_clone

# 3) Count per clone x condition, excluding largest clone
clone_cond <- seu_singlets@meta.data %>%
  filter(!is.na(CTgene)) %>%
  filter(CTgene != largest_clone) %>%
  group_by(CTgene, Condition, dual_TCR) %>%
  summarise(count = n(), .groups = "drop")

# 4) Filter clones by minimum count
min_count <- 3

clones_keep <- clone_cond %>%
  group_by(CTgene) %>%
  filter(max(count) >= min_count) %>%
  pull(CTgene) %>%
  unique()

clone_cond_filtered <- clone_cond %>%
  filter(CTgene %in% clones_keep) %>%
  group_by(CTgene) %>%
  mutate(
    total        = sum(count),
    is_dual      = any(dual_TCR == TRUE),
    count_capped = pmin(count, 50)
  ) %>%
  ungroup() %>%
  arrange(desc(is_dual), desc(total)) %>%
  mutate(CTgene = factor(CTgene, levels = unique(CTgene)))

# 5) Condition level order
cond_levels <- levels(factor(clone_cond_filtered$Condition))

# 6) Plot
p1 <- ggplot() +
  geom_point(
    data = filter(clone_cond_filtered, count > 1),
    aes(
      x    = CTgene,
      y    = Condition,
      size = count_capped,
      fill = count
    ),
    shape = 21,
    color = "white",
    stroke = 0
  ) +
  geom_point(
    data = filter(clone_cond_filtered, count > 1 & is_dual),
    aes(
      x    = CTgene,
      y    = Condition,
      size = count_capped
    ),
    shape = 21,
    fill = NA,
    color = "#2d2d2d",
    stroke = 0
  ) +
  scale_fill_gradientn(
    colours = c("#fde725", "#35b779", "#31688e", "#440154"),
    trans   = "log1p",
    name    = "Clone size\n(actual)",
    breaks  = c(1, 5, 10, 20, 40, 800),
    labels  = c("1", "5", "10","20", "40", "80")
  ) +
  scale_size_continuous(
    range  = c(2, 12),
    name   = "Clone size\n(capped at 50)",
    breaks = c(1, 5, 10, 25, 50),
    labels = c("1", "5", "10", "25", "≥50")
  ) +
  guides(
    fill = guide_colorbar(order = 1),
    size = guide_legend(
      order = 2,
      override.aes = list(fill = "#31688e", color = "white")
    )
  ) +
  scale_y_discrete(limits = cond_levels) +
  theme_minimal(base_size = 11) +
  theme(
    axis.text.x     = element_text(angle = 90, vjust = 0.5, hjust = 1, size = 6),
    axis.text.y     = element_text(size = 10),
    axis.title      = element_blank(),
    panel.grid      = element_blank(),
    legend.position = "right"
  ) +
  ggtitle(
    paste0(
      "Clonal expansion per condition\n",
      "largest clone removed: ", largest_clone
    )
  )

p1
ggsave(plot = p1, filename = paste0(fig_dir, "Fig6b_clone_sizes_per_population.pdf"),
       width = 9.5, height = 7.5)

## 9. Repertoire structure (supporting analysis, not in the manuscript)

scRepertoire metadata are added to the object (`combineExpression`), followed by clonal proportion,
diversity indices, clone-size categories and Morisita overlap between the populations.

In [ ]:
combined[[1]]$barcode <- gsub("^sample1_sample1_", "", combined[[1]]$barcode)
seu_singlets <- combineExpression(
  combined,
  seu_singlets,
  cloneCall = "strict"
)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 5)
clonalProportion(
  seu_singlets@meta.data,
  cloneCall = "strict",
  group = "Condition"
)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 5)
df <- list(seu_singlets@meta.data)

clonalDiversity(
  df,
  cloneCall = "strict",
  group = "Condition"
)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)
clonalHomeostasis(
  seu_singlets@meta.data,
  cloneCall = "strict",
  group = "Condition"
)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4)
df_list <- split(seu_singlets@meta.data, seu_singlets$Condition)
clonalOverlap(
  df_list,
  cloneCall = "strict",
  method = "morisita"
)

## 10. Projection onto the adoptive-transfer reference

Reference: annotated T cells without Unknowns. The reference UMAP is recomputed with `return.model = TRUE`
(umap-learn, correlation metric) so that query cells can be projected. Anchors on the reference PCA
(features = genes of the reference PCA present in the query), `MapQuery` transfers `Celltype_2`.

In [ ]:
original_gex <- readRDS(ref_rds)

In [ ]:
original_gex <- UpdateSeuratObject(original_gex)

In [ ]:
cells_keep <- levels(original_gex@meta.data$Celltype_2)[
  !levels(original_gex@meta.data$Celltype_2) %in% c("Unknown 1", "Unknown 2", "Unknown 3")
]

original_gex <- subset(
  original_gex,
  subset = Celltype_2 %in% cells_keep
)

# remove unused factor levels
original_gex$Celltype_2 <- droplevels(original_gex$Celltype_2)

In [ ]:
original_gex <- RunUMAP(
  original_gex,
  reduction = "pca",
  dims = 1:20,

  umap.method = "umap-learn",
  metric = "correlation",

  n.neighbors = 30,
  min.dist = 0.3,

  seed.use = 42,
  return.model = TRUE,

  reduction.name = "umap_python",
  reduction.key = "UMAPpython_",

  verbose = TRUE
)

In [ ]:
transfer_features <- intersect(
  rownames(Loadings(original_gex, reduction = "pca")),
  rownames(seu_singlets[["RNA"]])
)

length(transfer_features)   # 1,960 genes

In [ ]:
anchors <- FindTransferAnchors(
  reference = original_gex,
  query = seu_singlets,
  features = transfer_features,
  normalization.method = "LogNormalize",
  reference.reduction = "pca",
  reduction = "pcaproject",
  dims = 1:20,
  k.anchor = 5
)

In [ ]:
# Map labels and project
query_map <- MapQuery(
  anchorset = anchors,
  reference = original_gex,
  query = seu_singlets,
  refdata = list(
    Celltype_2 = "Celltype_2"
  ),
  reference.reduction = "pca",
  reduction.model = "umap_python"
)

DimPlot(
  query_map,
  reduction = "ref.umap",
  group.by = "predicted.Celltype_2",
  cols = celltype_colors_2
)

Reference UMAP (not in the manuscript; Fig. 2a is made in 04).

In [ ]:
p1 <- DimPlot(original_gex, reduction = "umap", group.by = "Celltype_2", cols = celltype_colors_3)
p1
ggsave(plot = p1, filename = paste0(fig_dir, "F_reference_UMAP.pdf"),
       width = 6, height = 4)

**Figure 5d** – query cells on the reference UMAP, coloured by transferred cell type.

In [ ]:
p1 <- DimPlot(query_map, reduction = "ref.umap",group.by = "predicted.Celltype_2", cols = celltype_colors_2)
p1
ggsave(plot = p1, filename = paste0(fig_dir, "Fig5d_refUMAP_predicted_celltypes.pdf"),
       width = 6, height = 4)

**Figure 5e** – query cells on the reference UMAP, coloured by population.

In [ ]:
p1 <- DimPlot(query_map, reduction = "ref.umap",group.by = "Condition", cols = cols_samples)
p1
ggsave(plot = p1, filename = paste0(fig_dir, "Fig5e_refUMAP_CD62L_KLF2_populations.pdf"),
       width = 6, height = 4)

**Figure 5f** – proportion of the four populations within each predicted cell type (all cells).

In [ ]:
# create dataframe
df <- as.data.frame(table(
  Reference = query_map$predicted.Celltype_2,
  SingletGroup = query_map$Condition
))

# define desired order
cell_order <- names(celltype_colors_3)

# apply order
df$Reference <- factor(df$Reference, levels = rev(cell_order))

p1 <- ggplot(df, aes(x = Reference, y = Freq, fill = SingletGroup)) +
  geom_bar(stat = "identity", position = "fill") +
scale_fill_manual(values = cols_samples) +
  theme_classic() +
  ylab("Proportion") +
  coord_flip()

p1
ggsave(plot = p1, filename = paste0(fig_dir, "Fig5f_celltypes_by_population_proportion.pdf"),
       width = 6, height = 3)

Same as cell counts (not in the manuscript).

In [ ]:
p1 <- ggplot(df, aes(x = Reference, y = Freq, fill = SingletGroup)) +
  geom_bar(stat = "identity", position = "stack") +
scale_fill_manual(values = cols_samples) +
  theme_classic() +
  ylab("Cell count") +
  coord_flip()

p1
ggsave(plot = p1, filename = paste0(fig_dir, "F_celltypes_by_population_counts.pdf"),
       width = 6, height = 3)

### Dual-TCR cells only

Proportions (not in the manuscript) and cell counts → **Figure 6c**.

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 3)

# subset only dual TCR cells
seu_dual <- subset(query_map, subset = dual_TCR == TRUE)

# create dataframe
df <- as.data.frame(table(
  Reference = seu_dual$predicted.Celltype_2,
  SingletGroup = seu_dual$Condition
))

# define desired order
cell_order <- names(celltype_colors_3)

# apply order
df$Reference <- factor(df$Reference, levels = cell_order)

# plot
p1 <- ggplot(df, aes(x = Reference, y = Freq, fill = SingletGroup)) +
  geom_bar(stat = "identity", position = "fill") +
  scale_fill_manual(values = cols_samples) +
  theme_classic(base_size = 14) +
  ylab("Proportion") +
  xlab("") + ggtitle("Only Dual TCRs") +
  coord_flip()

p1
ggsave(plot = p1, filename = paste0(fig_dir, "F_dualTCR_celltypes_by_population_proportion.pdf"),
       width = 6, height = 3)

In [ ]:
p1 <- ggplot(df, aes(x = Reference, y = Freq, fill = SingletGroup)) +
  geom_bar(stat = "identity", position = "stack") +
scale_fill_manual(values = cols_samples) +
  theme_classic() +
  ylab("Cell count") +
  coord_flip() + ggtitle("Only Dual TCRs")
p1
ggsave(plot = p1, filename = paste0(fig_dir, "Fig6c_dualTCR_celltypes_by_population_counts.pdf"),
       width = 6, height = 3)

Percentage of dual-TCR cells per population (numbers of Fig. 6a).

In [ ]:
dual_tcr_df <- seu_singlets@meta.data %>%
  group_by(Condition) %>%
  summarise(
    pct_dual_tcr = 100 * mean(dual_TCR, na.rm = TRUE),
    .groups = "drop"
  )

dual_tcr_df

## Session info

In [ ]:
sessionInfo()